### Extract structured data from Dermatologic Disease Database PDFs

#### Helpful links:
*   [Gemini SDK](https://cloud.google.com/vertex-ai/generative-ai/docs/reference/python/latest/vertexai.preview.generative_models)
*   [GCS SDK](https://cloud.google.com/python/docs/reference/storage/latest)

In [ ]:
!pip install vertexai
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 19.5 MB/s eta 0:00:00
  Attempting uninstall: google-cloud-aiplatform
    Found existing installation: google-cloud-aiplatform 1.63.0
    Uninstalling google-cloud-aiplatform-1.63.0:
      Successfully uninstalled google-cloud-aiplatform-1.63.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 1.8 MB/s eta 0:00:00


In [ ]:
import json, os
import time
from PyPDF2 import PdfReader, PdfWriter
from pathlib import Path
import vertexai
from vertexai.generative_models import GenerativeModel, Part, HarmCategory, HarmBlockThreshold
from google.cloud import storage
from google.cloud.storage import transfer_manager

project_id = "segfault-434120"
location = "us-central1"
bucket_name = "skincare-products"
raw_folder = "raw/dermatologic_diseases/raw/"       # files which are downloaded by download_tsa_reports.py are written into this folder
split_folder = "raw/dermatologic_diseases/split/"   # input location for the extract function
llm_folder = "raw/dermatologic_diseases/llm_text/"  # output location for the extract function
model_name = "gemini-1.5-flash-001"       # Note: Gemini Flash doesn't support the schema response option
prompt = """
INPUT:
You are recieving a disease overview in a pdf file. This file contains information centering around the primary disease specified in the title.

INSTRUCTIONS:
Convert the content into a single JSON object, focusing on summarizing and rephrasing the information regarding the primary disease. The disease JSON object must strictly adhere to the following schema:
{
  name: string
  physical_description: string
  causes: string
  treatments: string | null
}

SCHEMA EXPLANATION:
  name: Disease name
  physical_description: A brief, reworded physical description
  causes: The main causes, expressed concisely
  treatments: Ingredient-based treatments if available

SPECIAL-CASES:
If no ingredient-based treatments are mentioned, set the ingredient_treatments field to null. If there are ingredient-based treatments, list them, summarized with a comma delimiter.
Even if there are specific sub-categories of the disease mentioned, focus on the overarching main disease based on the title.

EXPECTED RESPONSE OUTPUT:
Avoid directly copying sentences from the original text, and instead focus on generating a clear and concise summary.
Only return a SINGLE disease json object about the primary disease in your response.

"""

def split_documents():

    storage_client = storage.Client()
    blobs = storage_client.list_blobs(bucket_name, prefix=raw_folder)

    for blob in blobs:

        if blob.name == raw_folder:
            continue

        source_filename = blob.name
        print("downloading", source_filename)

        # Ensure the local directory exists
        local_dir = os.path.dirname(source_filename)
        if not os.path.exists(local_dir):
            os.makedirs(local_dir)

        blob.download_to_filename(source_filename)

        start_page = 1
        pdf_reader = PdfReader(blob.name)
        pdf_writer = PdfWriter()

        file_name = blob.name.split(".pdf")[0].replace(raw_folder, split_folder)

        for page_num, page_data in enumerate(pdf_reader.pages, 1):
            pdf_writer.add_page(page_data)
            remainder = page_num % 500

            if (page_num % 500 == 0):
                file_path = f"{file_name}_{start_page}_{page_num}.pdf"
                print("trying to write", file_path)

                # Ensure the local directory exists
                split_local_dir = os.path.dirname(file_path)
                if not os.path.exists(split_local_dir):
                    os.makedirs(split_local_dir)

                with open(file_path, "wb") as out:
                    pdf_writer.write(out)
                    pdf_writer = PdfWriter()
                    print("wrote local file", file_path)

                # move the start page marker
                start_page = page_num + 1

        # write remaining file
        if page_num >= start_page:
            file_path = f"{file_name}_{start_page}_{page_num}.pdf"
            print("trying to write last file", file_path)

            # Ensure the local directory exists
            split_local_dir = os.path.dirname(file_path)
            if not os.path.exists(split_local_dir):
                os.makedirs(split_local_dir)

            with open(file_path, "wb") as out:
                pdf_writer.write(out)
                print("wrote last local file", file_path)


def copy_to_GCS(local_folder, gcs_folder, file_extension):
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    # Step 1: Clear out the existing files in the target GCS folder
    print(f"Clearing files from {gcs_folder} in bucket {bucket_name}...")
    blobs = storage_client.list_blobs(bucket_name, prefix=gcs_folder)

    for blob in blobs:
        if blob.name != gcs_folder:  # Skip the folder itself
            print(f"Deleting {blob.name}")
            blob.delete()

    # Step 2: Proceed with the file upload
    directory_as_path_obj = Path(local_folder)
    file_paths = directory_as_path_obj.rglob(file_extension)
    relative_paths = [path.relative_to(local_folder) for path in file_paths]
    string_paths = [str(path) for path in relative_paths]
    print("Found {} files.".format(string_paths))

    results = transfer_manager.upload_many_from_filenames(bucket, string_paths, source_directory=local_folder, blob_name_prefix=gcs_folder, max_workers=5)

    for name, result in zip(string_paths, results):

        if isinstance(result, Exception):
            print("Failed to upload {} due to exception: {}".format(name, result))
        else:
            print("Uploaded {} to {}.".format(name, bucket.name))


def extract(perform_refresh=False):

    vertexai.init(project=project_id, location=location)
    model = GenerativeModel(model_name)

    storage_client = storage.Client()
    blobs = storage_client.list_blobs(bucket_name, prefix=split_folder)

    for blob in blobs:

        if blob.name == split_folder:
            continue

        # check if file has already been processed
        filename = blob.name.replace(split_folder, llm_folder).replace(".pdf", ".txt")

        f = Path(filename)
        if f.exists() and not perform_refresh:
            print(f"{filename} already exists")
            continue

        print(f"extracting {blob.name}")
        file_content = Part.from_uri(f"gs://{bucket_name}/{blob.name}", "application/pdf")
        resp = model.generate_content(
            contents=[file_content, prompt],
            safety_settings={
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
            })
        resp_str = str(resp.candidates[0].text).replace("```json", "").replace("```", "")
        print("got resp from LLM")

        # format JSON into single line
        formatted_resp_str = resp_str.replace("\n", "").replace("\r", "")

        # Ensure the local directory exists
        llm_local_dir = os.path.dirname(filename)
        if not os.path.exists(llm_local_dir):
            os.makedirs(llm_local_dir)

        f = open(filename, "w")
        f.write(formatted_resp_str)
        f.close()
        print("wrote file", filename)


if __name__ == "__main__":
    split_documents() # split pdf documents due to large size
    copy_to_GCS(split_folder, split_folder, "*.pdf") # copy split documents to GCS
    extract(perform_refresh=True) # call LLM and extract attributes from documents
    copy_to_GCS(llm_folder, llm_folder, "*.txt") # copy LLM output to GCS


downloading raw/dermatologic_diseases/raw/ACANTHOMA_FISSURATUM.pdf
trying to write last file raw/dermatologic_diseases/split/ACANTHOMA_FISSURATUM_1_1.pdf
wrote last local file raw/dermatologic_diseases/split/ACANTHOMA_FISSURATUM_1_1.pdf
downloading raw/dermatologic_diseases/raw/ACANTHOSIS_NIGRICANS.pdf
trying to write last file raw/dermatologic_diseases/split/ACANTHOSIS_NIGRICANS_1_1.pdf
wrote last local file raw/dermatologic_diseases/split/ACANTHOSIS_NIGRICANS_1_1.pdf
downloading raw/dermatologic_diseases/raw/ACCESSORY_TRAGUS.pdf
trying to write last file raw/dermatologic_diseases/split/ACCESSORY_TRAGUS_1_1.pdf
wrote last local file raw/dermatologic_diseases/split/ACCESSORY_TRAGUS_1_1.pdf
downloading raw/dermatologic_diseases/raw/ACCUTANE.pdf
trying to write last file raw/dermatologic_diseases/split/ACCUTANE_1_2.pdf
wrote last local file raw/dermatologic_diseases/split/ACCUTANE_1_2.pdf
downloading raw/dermatologic_diseases/raw/ACNE.pdf
trying to write last file raw/dermatologic_diseas